# Tugas Praktikum Data Mining II TM7
# Social Media Mining

## Nama: Faiz Iqbal I'tishom
## NIM: 164231059
## Kelas: SD-A2

## Import Library dan Subset Data

In [1]:
# ============================
# 📦 Import Library Lengkap
# ============================

import os, re, json, random
from datetime import datetime
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)
from datasets import Dataset
import evaluate

import nltk
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import emoji

import networkx as nx
import community as community_louvain   # python-louvain
from sklearn.model_selection import train_test_split
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix

# ============================
# 📂 Load Dataset
# ============================

# Path dataset (pastikan nama file sesuai persis)
file_path = "/kaggle/input/data-data-mining-ii-tm7/Data_Data Mining II TM7.csv"

c:\Users\Faiz Iqbal\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\Faiz Iqbal\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\Faiz Iqbal\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


ModuleNotFoundError: No module named 'evaluate'

## Exploratory Data Analysis dan Pre-Processing

In [ ]:
print("=== Informasi Data ===")
print(df.info())
print("\n=== Missing Value ===")
print(df.isna().sum())

# --- 4. Filter Tweet Bahasa Indonesia ---
df_id = df[df['lang'] == 'in'][['created_at', 'full_text']].copy()
print(f"\nJumlah tweet berbahasa Indonesia: {len(df_id)}")

# --- 5. Konversi Kolom Waktu ---
df_id['created_at'] = pd.to_datetime(df_id['created_at'], errors='coerce')
df_id.dropna(subset=['created_at'], inplace=True)

In [ ]:
# =========================================================
#   ADVANCED TEXT PREPROCESSING
# =========================================================

# --- Siapkan Stopwords dan Stemmer ---
ind_stopwords = stopwords.words('indonesian')
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# --- Fungsi Pembersihan Lengkap ---
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www.\S+', '', text)                # hapus URL
    text = re.sub(r'@\w+|#\w+', '', text)                      # hapus mention & hashtag
    text = re.sub(r'\d+', '', text)                            # hapus angka
    text = text.translate(str.maketrans('', '', string.punctuation))  # hapus tanda baca
    text = re.sub(r'\s+', ' ', text).strip()                   # hapus spasi berlebih
    return text

def preprocess_text(text):
    text = clean_text(text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in ind_stopwords and len(w) > 2]
    stemmed = [stemmer.stem(w) for w in tokens]
    return ' '.join(stemmed)

# --- Terapkan ke Dataset ---
print("\nMembersihkan teks, ini mungkin memakan waktu...")
df_id['clean_text'] = df_id['full_text'].astype(str).apply(preprocess_text)
print("Selesai membersihkan teks!")

# --- Tambahkan Fitur Panjang Teks ---
df_id['text_length'] = df_id['clean_text'].apply(lambda x: len(x.split()))


Membersihkan teks, ini mungkin memakan waktu...


KeyboardInterrupt: 

In [ ]:
# =========================================================
#   ANALISIS DESKRIPTIF
# =========================================================

print("\n=== Statistik Deskriptif Panjang Teks ===")
print(df_id['text_length'].describe())

plt.figure(figsize=(8,5))
sns.histplot(df_id['text_length'], bins=40, kde=True, color='skyblue')
plt.title("Distribusi Panjang Komentar (Jumlah Kata)")
plt.xlabel("Jumlah Kata per Komentar")
plt.ylabel("Frekuensi")
plt.show()

In [ ]:
# =========================================================
#   WORD CLOUD & TOP WORDS
# =========================================================

all_words = ' '.join(df_id['clean_text'])
wordcloud = WordCloud(width=1000, height=600, background_color='white', collocations=False).generate(all_words)

plt.figure(figsize=(12,7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Word Cloud - Semua Komentar Bahasa Indonesia", fontsize=16)
plt.show()

# --- Top Words ---
words = all_words.split()
word_freq = Counter(words)
top_words = pd.DataFrame(word_freq.most_common(20), columns=['Word', 'Frequency'])

plt.figure(figsize=(10,6))
sns.barplot(y='Word', x='Frequency', data=top_words, palette='viridis')
plt.title("20 Kata Teratas dalam Komentar")
plt.xlabel("Frekuensi")
plt.ylabel("Kata")
plt.show()

In [ ]:
# =========================================================
#   ANALISIS WAKTU
# =========================================================

# --- Buat Kolom Tanggal & Jam ---
df_id['date'] = df_id['created_at'].dt.date
df_id['hour'] = df_id['created_at'].dt.hour

# --- Aktivitas Harian ---
daily_tweets = df_id.groupby('date').size().reset_index(name='count')

plt.figure(figsize=(12,6))
sns.lineplot(x='date', y='count', data=daily_tweets, color='orange')
plt.title("Tren Aktivitas Tweet Harian")
plt.xlabel("Tanggal")
plt.ylabel("Jumlah Tweet")
plt.grid(True)
plt.show()

# --- Aktivitas per Jam (Countplot) ---
plt.figure(figsize=(10,5))
sns.countplot(x='hour', data=df_id, palette='coolwarm')
plt.title("Distribusi Aktivitas Tweet Berdasarkan Jam")
plt.xlabel("Jam (0–23)")
plt.ylabel("Jumlah Tweet")
plt.show()

# --- Boxplot Aktivitas per Jam (Distribusi Panjang Komentar) ---
plt.figure(figsize=(12,6))
sns.boxplot(x='hour', y='text_length', data=df_id, palette='crest')
plt.title("Distribusi Panjang Komentar Berdasarkan Jam (Boxplot)")
plt.xlabel("Jam (0–23)")
plt.ylabel("Jumlah Kata per Komentar")
plt.show()

In [ ]:
# =========================================================
#   ANALISIS BERDASARKAN PERIODE WAKTU
# =========================================================

df_id['month'] = df_id['created_at'].dt.to_period('M')

# --- Top Words per Bulan ---
monthly_top = (
    df_id.groupby('month')['clean_text']
    .apply(lambda x: ' '.join(x).split())
    .apply(lambda words: Counter(words).most_common(5))
)

monthly_top_df = pd.DataFrame({
    'Month': monthly_top.index.astype(str),
    'Top Words': monthly_top.values
})
print("\n=== Top 5 Words per Bulan ===")
print(monthly_top_df)

# --- Jumlah Tweet per Bulan ---
monthly_count = df_id.groupby('month').size().reset_index(name='count')

plt.figure(figsize=(10,5))
sns.barplot(x='month', y='count', data=monthly_count, palette='magma')
plt.title("Jumlah Tweet per Bulan")
plt.xlabel("Bulan")
plt.ylabel("Jumlah Tweet")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# =========================================================
#   IDE TAMBAHAN UNTUK INSIGHT
# =========================================================

# 1️⃣ Analisis waktu aktif pengguna (pagi, siang, sore, malam)
df_id['time_of_day'] = pd.cut(
    df_id['hour'],
    bins=[0,6,12,18,24],
    labels=['Malam', 'Pagi', 'Siang', 'Sore'],
    right=False
)
avg_length_time = df_id.groupby('time_of_day')['text_length'].mean().reset_index()

plt.figure(figsize=(8,5))
sns.barplot(x='time_of_day', y='text_length', data=avg_length_time, palette='Blues')
plt.title("Rata-rata Panjang Komentar Berdasarkan Waktu dalam Sehari")
plt.xlabel("Waktu")
plt.ylabel("Rata-rata Jumlah Kata")
plt.show()

# 2️⃣ Korelasi antara panjang teks dan volume tweet per jam
hourly_stats = df_id.groupby('hour')['text_length'].agg(['mean', 'count']).reset_index()

fig, ax1 = plt.subplots(figsize=(12,6))
sns.lineplot(x='hour', y='count', data=hourly_stats, color='tab:blue', ax=ax1)
ax1.set_ylabel("Jumlah Tweet", color='tab:blue')
ax2 = ax1.twinx()
sns.lineplot(x='hour', y='mean', data=hourly_stats, color='tab:red', ax=ax2)
ax2.set_ylabel("Rata-rata Panjang Teks", color='tab:red')
plt.title("Hubungan Antara Volume dan Panjang Teks per Jam")
plt.show()

## Sentiment Analysis

In [ ]:
# ==========================
#  Setup - install libs (run once)
# ==========================
# jalankan ini di terminal / notebook cell (jangan jalankan berulang kalo sudah terinstall)
# !pip install transformers datasets emoji Sastrawi nltk wordcloud scikit-learn matplotlib seaborn

# ==========================
#  Import & util functions
# ==========================


# make plots prettier
sns.set(style="whitegrid")
nltk.download('stopwords')

# ==========================
#  Load data (edit path sesuai)
# ==========================
DATA_PATH = r"/kaggle/input/data-data-mining-ii-tm7/Data_Data Mining II TM7.csv"
df = pd.read_csv(DATA_PATH)
print("Columns:", df.columns.tolist())
print(df.shape)
# Asumsi kolom: 'created_at', 'full_text', 'lang', dsb — jika beda, sesuaikan.

# Fokus hanya tweet bahasa Indonesia
if 'lang' in df.columns:
    df_id = df[df['lang'] == 'in'][['created_at','full_text']].copy()
else:
    df_id = df[['created_at','full_text']].copy()

# parse created_at
df_id['created_at'] = pd.to_datetime(df_id['created_at'], errors='coerce')
df_id = df_id.dropna(subset=['created_at']).reset_index(drop=True)


In [ ]:
# ==========================
#  Advanced preprocessing (improved)
# ==========================
# prepare resources
ind_stopwords = set(stopwords.words('indonesian'))  # may need custom additions
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# sample slang dictionary (tambah sendiri sesuai domain)
SLANG_DICT = {
    'ga': 'tidak', 'gak': 'tidak', 'tdk': 'tidak', 'g': 'tidak',
    'bgd':'banget','bgt':'banget','btw':'omong-omong',
    'klo':'kalau','kalo':'kalau','yg':'yang'
}

NEGATIONS = set(['tidak','gak','ga','bukan','tdk','tak','jangan'])
INTENSIFIERS = {'sangat':2.0, 'banget':1.7, 'sekali':1.6, 'amat':1.6, 'terlalu':1.5}

# emoticon map (simple)
EMOTICON_MAP = {
    ':)': 'senang', ':-)': 'senang', ':(': 'sedih', ':-(': 'sedih',
    ':D': 'senang', ';)': 'senang', ':/': 'ragu'
}

def normalize_repeated_chars(text):
    # abaikan reduplikasi huruf berlebihan: helloooo -> helloo
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

def text_preprocess_full(text, do_stem=True):
    text = str(text)
    # demojize -> convert emoji to text like :smiling_face:
    text = emoji.demojize(text, delimiters=(' ', ' '))
    # replace emoticons
    for k,v in EMOTICON_MAP.items():
        text = text.replace(k, ' '+v+' ')
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)   # remove urls
    text = re.sub(r'@\w+', ' ', text)              # remove mentions
    text = re.sub(r'#', ' ', text)                 # remove hash symbol but keep text
    text = re.sub(r'\d+', ' ', text)               # remove numbers
    text = normalize_repeated_chars(text)
    # remove punctuation (but emoticon words already replaced)
    text = text.translate(str.maketrans('', '', string.punctuation))
    # tokenize
    tokens = [t for t in text.split() if t.strip()]
    # slang normalization
    tokens = [SLANG_DICT.get(t,t) for t in tokens]
    # remove stopwords & short tokens
    tokens = [t for t in tokens if t not in ind_stopwords and len(t)>1]
    # optional stemming
    if do_stem:
        tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

# apply preprocessing (this may take some time for big datasets)
print("Preprocessing texts (this may take a while)...")
df_id['clean_text'] = df_id['full_text'].astype(str).apply(text_preprocess_full)
df_id['text_length'] = df_id['clean_text'].apply(lambda x: len(x.split()))
print("Done preprocessing. Examples:")
print(df_id[['full_text','clean_text']].head(3))

In [ ]:
# ==========================
#  1) Lexicon-based method (using InSet lexicon)
# ==========================
# Strategy:
# - Load InSet lexicon if available (Hugging Face dataset or GitHub)
# - Build dict word -> score (if weights present) or +1/-1
# - Score sentence with negation & intensifier handling

# Try to load InSet from HuggingFace datasets (fallback: minimal internal lexicon)
try:
    from datasets import load_dataset
    inset_ds = load_dataset("SEACrowd/inset_lexicon")
    # inspect structure to load word->score mapping
    # dataset might contain multiple splits or a 'train' split; try to coalesce
    lex_dict = {}
    for split in inset_ds.keys():
        for row in inset_ds[split]:
            # expected fields: 'word','score' or 'label' — handle common variants
            if 'word' in row and ('score' in row or 'label' in row):
                word = row['word']
                score = row.get('score', row.get('label', 0))
                try:
                    score = float(score)
                except:
                    score = 1.0 if str(score).lower() in ['pos','positive','1'] else -1.0
                lex_dict[word] = score
    if len(lex_dict)==0:
        raise Exception("inset dataset loaded but lexicon empty")
    print("Loaded InSet lexicon from HuggingFace with", len(lex_dict), "entries.")
except Exception as e:
    print("Could not load InSet automatically (error: {}). Falling back to a minimal sample lexicon.".format(e))
    # small sample lexicon; replace by full InSet by download
    lex_dict = {'baik':2.0, 'bagus':2.0, 'senang':1.5, 'jelek':-2.0, 'buruk':-2.0, 'benci':-2.5, 'cinta':2.5}
    print("Using sample lexicon of length", len(lex_dict))
    print("Tip: download InSet lexicon (https://github.com/fajri91/InSet) or load 'SEACrowd/inset_lexicon' via datasets to use full lexicon.")

# scoring function with negation & intensifier handling
def lexicon_score(text, lexicon=lex_dict, negations=NEGATIONS, intensifiers=INTENSIFIERS, neg_window=3):
    tokens = text.split()
    score = 0.0
    for i, tok in enumerate(tokens):
        if tok in lexicon:
            tok_score = float(lexicon.get(tok, 0.0))
            # check for negation in the previous neg_window tokens
            start = max(0, i-neg_window)
            window = tokens[start:i]
            neg = any(w in negations for w in window)
            # intensifier check: look at previous 2 tokens and pick max multiplier
            prev_tokens = tokens[max(0,i-2):i]
            intens = 1.0
            for t in prev_tokens:
                intens = max(intens, intensifiers.get(t,1.0))
            if neg:
                tok_score = -tok_score
            tok_score = tok_score * intens
            score += tok_score
    return score

def lexicon_label_from_score(score, pos_thresh=0.5, neg_thresh=-0.5):
    if score >= pos_thresh:
        return 'positive'
    elif score <= neg_thresh:
        return 'negative'
    else:
        return 'neutral'

# compute lexicon scores & labels
print("Computing lexicon-based sentiment scores ...")
df_id['lex_score'] = df_id['clean_text'].apply(lambda t: lexicon_score(t, lexicon=lex_dict))
df_id['lex_sentiment'] = df_id['lex_score'].apply(lexicon_label_from_score)

# quick distribution
print(df_id['lex_sentiment'].value_counts())

In [ ]:
# ==========================
#  2) Transformer-based method (Hugging Face pipeline)
# ==========================
# Choose model(s) -- contoh model yang ada di HF:
# - "mdhugol/indonesia-bert-sentiment-classification" (general sentiment)
# - "Aardiiiiy/indobertweet-base-Indonesian-sentiment-analysis" (Twitter-specific)
# kamu bisa ganti model_name sesuai kebutuhan

model_name = "mdhugol/indonesia-bert-sentiment-classification"  # contoh
# alternatif_twitter = "Aardiiiiy/indobertweet-base-Indonesian-sentiment-analysis"

# device: 0=GPU, -1=CPU
import torch
device = 0 if torch.cuda.is_available() else -1
print("Using device:", device)

# initialize pipeline
try:
    sentiment_pipe = pipeline(
        task="text-classification",
        model=model_name,
        tokenizer=model_name,
        device=device if device>=0 else -1,
        return_all_scores=True
    )
    print("Transformer model loaded:", model_name)
except Exception as e:
    print("Gagal load transformer model:", e)
    sentiment_pipe = None

# helper to map pipeline output (return_all_scores=True) to label string 'positive/neutral/negative'
def pipe_predict_label(pipe_output):
    # pipe_output is list of list-of-dicts (if batched), or list-of-dicts per item
    # handle if single item vs batch
    def single_to_label(scores):
        # scores: [{'label': 'LABEL_0', 'score':0.9}, ...] OR [{'label':'NEGATIVE','score':0.8},...]
        best = max(scores, key=lambda x: x['score'])
        lab = best['label'].lower()
        # map common patterns
        if lab in ['positive','neg','negative','neutral']:
            if 'pos' in lab:
                return 'positive'
            if 'neg' in lab:
                return 'negative'
            if 'neutral' in lab:
                return 'neutral'
        # fallback: if labels are LABEL_0 etc. -> try infer mapping by ordering (some models: [neg, neu, pos])
        # risky: we'll attempt heuristics: if three classes present and labels include '0','1','2'
        # return mapping by highest score index -> we will map label names to pos/neu/neg by checking dataset card if available.
        return lab  # return raw if unknown
    # if batch
    if isinstance(pipe_output, list) and isinstance(pipe_output[0], list):
        return [single_to_label(item) for item in pipe_output]
    elif isinstance(pipe_output, list) and isinstance(pipe_output[0], dict):
        # pipeline with return_all_scores might give for single item a list of dicts
        return single_to_label(pipe_output)
    else:
        return None

# run inference in batches (safely)
if sentiment_pipe is not None:
    texts = df_id['clean_text'].tolist()
    batch_size = 32
    preds = []
    print("Running transformer inference in batches (batch_size={})...".format(batch_size))
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        out = sentiment_pipe(batch)
        # out is list (batch) of list-of-dicts
        for scores in out:
            # pick best label
            best = max(scores, key=lambda x: x['score'])
            lab = best['label'].lower()
            # normalize common label names
            if 'pos' in lab:
                preds.append('positive')
            elif 'neg' in lab:
                preds.append('negative')
            elif 'neu' in lab:
                preds.append('neutral')
            else:
                # fallback: sometimes labels are 'LABEL_0'--we use the highest score label text raw
                preds.append(best['label'].lower())
    # store
    df_id['tf_sentiment'] = preds
else:
    df_id['tf_sentiment'] = None
    print("Transformer predictions not available.")

In [ ]:
# ==========================
#  3) Evaluation: If ground-truth labels exist, compute metrics; else compute inter-method agreement + export sample for manual labeling
# ==========================
# Detect ground truth column names commonly used
possible_label_cols = [c for c in df.columns if c.lower() in ['sentiment','label','sentimen','polarity','y']]
has_label = False
label_col = None
for c in possible_label_cols:
    if c in df.columns:
        label_col = c
        has_label = True
        break

# If no label found in original df, check df_id (maybe user already added label)
for c in df_id.columns:
    if c.lower() in ['sentiment','label','sentimen','polarity','y','gold']:
        label_col = c
        has_label = True
        break

if has_label:
    print("Ground truth label column detected:", label_col)
    # map labels to canonical form
    def norm_label(x):
        if pd.isna(x): return None
        s = str(x).lower()
        if any(t in s for t in ['pos','+','positif','positif','puas','bagus']):
            return 'positive'
        if any(t in s for t in ['neg','-','negatif','tidak suka','jelek','buruk','benci']):
            return 'negative'
        if any(t in s for t in ['neu','net','netral','neutral','netral']):
            return 'neutral'
        return s
    df_id['gold_label'] = df_id[label_col].apply(norm_label)
    df_eval = df_id.dropna(subset=['gold_label']).copy()
    print("Evaluable instances (with gold labels):", len(df_eval))

    # evaluate lexicon
    if 'lex_sentiment' in df_eval.columns:
        y_true = df_eval['gold_label'].tolist()
        y_pred_lex = df_eval['lex_sentiment'].tolist()
        print("=== Lexicon-based Evaluation ===")
        print(classification_report(y_true, y_pred_lex, digits=4))
        print("Confusion matrix (lexicon vs gold):")
        cm = confusion_matrix(y_true, y_pred_lex, labels=['positive','neutral','negative'])
        sns.heatmap(cm, annot=True, fmt='d', xticklabels=['pos','neu','neg'], yticklabels=['pos','neu','neg'])
        plt.title("Confusion Matrix: Lexicon vs Gold")
        plt.xlabel("Pred Lexicon"); plt.ylabel("Gold")
        plt.show()

    # evaluate transformer
    if 'tf_sentiment' in df_eval.columns:
        y_pred_tf = df_eval['tf_sentiment'].tolist()
        print("=== Transformer-based Evaluation ===")
        print(classification_report(y_true, y_pred_tf, digits=4))
        cm2 = confusion_matrix(y_true, y_pred_tf, labels=['positive','neutral','negative'])
        sns.heatmap(cm2, annot=True, fmt='d', xticklabels=['pos','neu','neg'], yticklabels=['pos','neu','neg'])
        plt.title("Confusion Matrix: Transformer vs Gold")
        plt.xlabel("Pred Transformer"); plt.ylabel("Gold")
        plt.show()

    # compare lexicon vs transformer agreement
    if 'lex_sentiment' in df_eval.columns and 'tf_sentiment' in df_eval.columns:
        mask = df_eval['lex_sentiment'].notna() & df_eval['tf_sentiment'].notna()
        kappa = cohen_kappa_score(df_eval.loc[mask,'lex_sentiment'], df_eval.loc[mask,'tf_sentiment'])
        print("Cohen's kappa between lexicon & transformer:", kappa)
else:
    print("No ground-truth labels found in dataset.")
    # Evaluate agreement between lexicon & transformer as proxy
    if ('lex_sentiment' in df_id.columns) and ('tf_sentiment' in df_id.columns) and df_id['tf_sentiment'].notna().all():
        comp = df_id[['clean_text','lex_sentiment','tf_sentiment']].copy()
        comp['agree'] = comp['lex_sentiment'] == comp['tf_sentiment']
        agree_rate = comp['agree'].mean()
        print(f"Agreement between lexicon & transformer: {agree_rate:.4f} ({comp.shape[0]} samples)")
        print("Cohen's kappa:", cohen_kappa_score(comp['lex_sentiment'], comp['tf_sentiment']))
        # show confusion matrix between methods
        cm = confusion_matrix(comp['lex_sentiment'], comp['tf_sentiment'], labels=['positive','neutral','negative'])
        sns.heatmap(cm, annot=True, fmt='d', xticklabels=['TF_pos','TF_neu','TF_neg'], yticklabels=['Lex_pos','Lex_neu','Lex_neg'])
        plt.title("Confusion Matrix: Lexicon (rows) vs Transformer (cols)")
        plt.show()
        # sample disagreements for manual inspection
        disag = comp[comp['agree']==False].sample(min(20,len(comp[comp['agree']==False])), random_state=42)
        print("Contoh tweet dengan disagreement (lex vs tf):")
        display(disag[['clean_text','lex_sentiment','tf_sentiment']].head(20))
    # export sample for manual labeling if user wants to create a small gold set
    SAMPLE_N = min(500, len(df_id))
    sample_for_labeling = df_id.sample(SAMPLE_N, random_state=42)[['created_at','full_text','clean_text']].reset_index(drop=True)
    out_path = "sample_for_manual_labeling.csv"
    sample_for_labeling.to_csv(out_path, index=False)
    print(f"Saved sample {SAMPLE_N} rows to '{out_path}' for manual labeling. Setelah dilabel, letakkan kolom 'gold_label' (positive/neutral/negative) lalu jalankan evaluasi.")

In [ ]:
# ==========================
#  4) Visualisasi lengkap (distribusi, tren, top words per sentiment)
# ==========================
# Ensure we have predictions; if missing transformer preds, skip those plots
def plot_sentiment_distribution(df, col, title_suffix=''):
    counts = df[col].value_counts().reindex(['positive','neutral','negative']).fillna(0)
    plt.figure(figsize=(6,4))
    sns.barplot(x=counts.index, y=counts.values, palette=['green','gray','red'])
    plt.title(f"Distribusi Sentimen ({col}) {title_suffix}")
    plt.ylabel("Jumlah")
    plt.show()
    # pie
    plt.figure(figsize=(6,6))
    plt.pie(counts.values, labels=counts.index, autopct='%1.1f%%', startangle=140)
    plt.title(f"Pie Sentimen ({col}) {title_suffix}")
    plt.show()

# lexicon distribution
if 'lex_sentiment' in df_id.columns:
    plot_sentiment_distribution(df_id, 'lex_sentiment', '(Lexicon)')

# transformer distribution
if 'tf_sentiment' in df_id.columns and df_id['tf_sentiment'].notna().all():
    plot_sentiment_distribution(df_id, 'tf_sentiment', '(Transformer)')

# trend over time: group by date and sentiment
for col in ['lex_sentiment','tf_sentiment']:
    if col in df_id.columns and df_id[col].notna().all():
        df_id['date'] = df_id['created_at'].dt.date
        trend = df_id.groupby(['date', col]).size().unstack(fill_value=0)
        # ensure columns order
        for c in ['positive','neutral','negative']:
            if c not in trend.columns:
                trend[c] = 0
        trend = trend[['positive','neutral','negative']]
        trend.plot(kind='line', figsize=(14,5), linewidth=2)
        plt.title(f"Tren Sentimen per Hari ({col})")
        plt.ylabel("Jumlah Tweet")
        plt.xlabel("Tanggal")
        plt.show()

# top words per sentiment (bar & wordcloud)
def top_words_for_sentiment(df, sentiment_col, top_n=20):
    res = {}
    for s in ['positive','neutral','negative']:
        texts = df[df[sentiment_col]==s]['clean_text'].dropna().tolist()
        if len(texts)==0:
            res[s] = []
            continue
        all_words = ' '.join(texts).split()
        freq = Counter(all_words)
        res[s] = freq.most_common(top_n)
    return res

if 'lex_sentiment' in df_id.columns:
    lex_top = top_words_for_sentiment(df_id, 'lex_sentiment', top_n=20)
    for s, lst in lex_top.items():
        print(f"Top words ({s}) - Lexicon-based:")
        print(lst)
        # barplot
        if lst:
            df_tmp = pd.DataFrame(lst, columns=['word','freq']).sort_values('freq', ascending=True)
            plt.figure(figsize=(6,4))
            sns.barplot(x='freq', y='word', data=df_tmp, palette='Blues_r')
            plt.title(f"Top words ({s}) - Lexicon")
            plt.show()
            # wordcloud
            wc = WordCloud(width=800, height=400, background_color='white').generate(' '.join([w for w,_ in lst for _ in range(int(_/1 if _>0 else 1))]))
            plt.figure(figsize=(10,5)); plt.imshow(wc, interpolation='bilinear'); plt.axis('off'); plt.show()

if 'tf_sentiment' in df_id.columns and df_id['tf_sentiment'].notna().all():
    tf_top = top_words_for_sentiment(df_id, 'tf_sentiment', top_n=20)
    for s, lst in tf_top.items():
        print(f"Top words ({s}) - Transformer-based:")
        print(lst)
        if lst:
            df_tmp = pd.DataFrame(lst, columns=['word','freq']).sort_values('freq', ascending=True)
            plt.figure(figsize=(6,4))
            sns.barplot(x='freq', y='word', data=df_tmp, palette='Greens_r')
            plt.title(f"Top words ({s}) - Transformer")
            plt.show()

# sentiment by hour heatmap
if 'tf_sentiment' in df_id.columns and df_id['tf_sentiment'].notna().all():
    df_id['hour'] = df_id['created_at'].dt.hour
    heat = df_id.groupby(['hour','tf_sentiment']).size().unstack(fill_value=0)
    # reorder columns
    for c in ['positive','neutral','negative']:
        if c not in heat.columns: heat[c]=0
    heat = heat[['positive','neutral','negative']]
    plt.figure(figsize=(10,5))
    sns.heatmap(heat.T, annot=True, fmt='d', cmap='YlOrRd')
    plt.title("Heatmap: Sentimen per Jam (Transformer)")
    plt.ylabel("Sentiment"); plt.xlabel("Hour of day")
    plt.show()

# boxplot: distribution of text_length per hour (already requested earlier)
plt.figure(figsize=(14,5))
sns.boxplot(x='hour', y='text_length', data=df_id)
plt.title("Distribusi Panjang Komentar (jumlah kata) per jam")
plt.show()

In [ ]:
# ==========================
#  5) Comparison summary & tips
# ==========================
print("Selesai. Ringkasan:")
print("- Lexicon-based: cepat, transparan, bagus sebagai baseline. Unggul jika domain kata ada di lexicon (contoh: InSet).")
print("- Transformer-based: lebih kontekstual, lebih baik menangani negasi kompleks / frasa, tapi memerlukan resource & model yang sudah fine-tuned.")

# Jika kamu ingin, saya bisa:
# - bantu download InSet full lexicon dan masukkan ke folder kerja (jika izinkan akses internet)
# - bantu fine-tune IndoBERT pada sampel labelled data kamu (jika kamu punya label)
# - lanjutkan ke Social Network Analysis dengan label sentimen ini (mention/retweet graph)

## Social Network Analysis

In [ ]:
# ============================================================
# FINE-TUNE INDOBERT + ASPECT-BASED SENTIMENT + SNA PIPELINE
# ============================================================
# Instal paket (jalankan sekali)
# !pip install transformers datasets accelerate tokenizers sentencepiece \
#              Sastrawi emoji nltk scikit-learn pandas matplotlib seaborn \
#              wordcloud networkx python-louvain gensim pyldavis tqdm

nltk.download('stopwords')
from nltk.corpus import stopwords

sns.set(style="whitegrid")

# -------------------------
# 1) Load & basic checks
# -------------------------
DATA_PATH = r"C:/Uner/Semester 5/Data Mining II/Coolyeah/TM7/Data_Data Mining II TM7.csv"
df_raw = pd.read_csv(DATA_PATH)
print("Columns:", df_raw.columns.tolist())
print("Rows:", len(df_raw))

# expected columns from earlier steps: 'created_at','full_text','clean_text', maybe 'gold_label', maybe user columns
# Use df_preproc as main working df
if 'clean_text' in df_raw.columns:
    df = df_raw.copy()
else:
    # if clean_text not present, try to create using earlier preprocess function (light)
    print("Kolom 'clean_text' tidak ditemukan. Menjalankan preprocessing singkat...")
    # light preprocess
    import string
    def simple_clean(t):
        t = str(t).lower()
        t = re.sub(r'http\S+|www.\S+', ' ', t)
        t = re.sub(r'@\w+', ' ', t)
        t = re.sub(r'#', ' ', t)
        t = re.sub(r'\d+',' ',t)
        t = t.translate(str.maketrans('','',string.punctuation))
        t = re.sub(r'\s+',' ',t).strip()
        return t
    df = df_raw.copy()
    df['clean_text'] = df['full_text'].astype(str).apply(simple_clean)

# parse created_at
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
df = df.dropna(subset=['created_at']).reset_index(drop=True)

# Detect label column (gold)
label_col = None
for c in df.columns:
    if c.lower() in ['gold_label','label','sentiment','sentimen','y']:
        label_col = c
        break

if label_col:
    print("Label column detected:", label_col)
    # normalize labels to {positive,neutral,negative}
    def norm_label(x):
        if pd.isna(x): return None
        s = str(x).lower()
        if any(k in s for k in ['pos','positif','positive','+','1']):
            return 'positive'
        if any(k in s for k in ['neg','negatif','negative','-','-1']):
            return 'negative'
        if any(k in s for k in ['neu','netral','neutral','0']):
            return 'neutral'
        return None
    df['gold_label'] = df[label_col].apply(norm_label)
else:
    print("Tidak ditemukan kolom label/gold di dataset.")
    # create sample for manual labeling (if not exist)
    sample_path = "sample_for_manual_labeling_finetune.csv"
    SAMPLE_N = min(800, len(df))  # produce up to 800 sample
    df.sample(SAMPLE_N, random_state=42)[['created_at','full_text','clean_text']].to_csv(sample_path, index=False)
    print(f"File sample untuk pelabelan manual disimpan ke: {sample_path}")
    print("Silakan beri kolom 'gold_label' dengan nilai {positive, neutral, negative} pada file tersebut lalu gabungkan kembali ke dataset, atau upload file labeled dan jalankan ulang pipeline.")
    # stop here since fine-tune needs labels
    raise SystemExit("Pipeline dihentikan: dataset belum memiliki gold_label. Isi label pada sample yang diekspor, gabungkan, dan jalankan ulang.")

# remove rows with missing gold_label
df_labelled = df.dropna(subset=['gold_label']).copy()
print("Jumlah data berlabel tersedia untuk fine-tune:", len(df_labelled))
if len(df_labelled) < 100:
    print("Perhatian: jumlah label < 100. Hasil fine-tune mungkin kurang stabil. Usahakan >500 contoh.")

In [ ]:
# -------------------------
# 2) Prepare dataset for HuggingFace Trainer
# -------------------------
# choose model base: IndoBERT or IndoBERTweet (twitter domain)
MODEL_NAME = "indobenchmark/indobert-base-p1"  # ganti ke model lain jika mau (mis. indobenchmark/indobertweet)
# jika ada model khusus twitter: "cahya/bert-base-indonesian-twitter" atau lainnya

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# label mapping
labels = sorted(df_labelled['gold_label'].unique())
label2id = {lab:i for i,lab in enumerate(labels)}
id2label = {i:lab for lab,i in label2id.items()}
print("Labels:", labels, label2id)

# tokenization function
def tokenize_fn(batch):
    return tokenizer(batch['clean_text'], truncation=True, padding='max_length', max_length=128)

# convert to datasets.Dataset
hf_ds = Dataset.from_pandas(df_labelled[['clean_text','gold_label']].rename(columns={'gold_label':'label'}))
# map label to ids
def map_label(example):
    example['label'] = label2id[example['label']]
    return example
hf_ds = hf_ds.map(map_label)
hf_ds = hf_ds.train_test_split(test_size=0.15, stratify_by_column='label', seed=42)
hf_ds = hf_ds.map(lambda examples: tokenizer(examples['clean_text'], truncation=True, padding='max_length', max_length=128), batched=True)
hf_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])
train_ds = hf_ds['train']
eval_ds = hf_ds['test']
print(train_ds, eval_ds)

In [ ]:
# -------------------------
# 3) Build model & Trainer
# -------------------------
num_labels = len(labels)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id)

# training args (adjust epochs/batch sizes for GPU/CPU)
output_dir = "./indobert_finetuned_sentiment"
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),   # use mixed precision if GPU available
    save_total_limit=2
)

# metric function (compute accuracy, precision, recall, f1 macro)
metric_acc = load_metric("accuracy")
from sklearn.metrics import f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=preds, references=labels)['accuracy']
    precision = precision_score(labels, preds, average='macro', zero_division=0)
    recall = recall_score(labels, preds, average='macro', zero_division=0)
    f1 = f1_score(labels, preds, average='macro', zero_division=0)
    return {"accuracy": acc, "precision_macro": precision, "recall_macro": recall, "f1_macro": f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics
)

In [ ]:
# -------------------------
# 4) Train (fine-tune)
# -------------------------
trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("Model terlatih disimpan di:", output_dir)

In [ ]:
# -------------------------
# 5) Evaluate on eval set & full labelled set
# -------------------------
eval_res = trainer.evaluate()
print("Eval metrics:", eval_res)

# predict on whole labelled dataset for detailed report
all_texts = df_labelled['clean_text'].tolist()
enc = tokenizer(all_texts, truncation=True, padding=True, max_length=128, return_tensors='pt')
device = 0 if torch.cuda.is_available() else -1
model.to('cuda' if torch.cuda.is_available() else 'cpu')
with torch.no_grad():
    outputs = model(enc['input_ids'].to(model.device), attention_mask=enc['attention_mask'].to(model.device))
    logits = outputs.logits.cpu().numpy()
preds = np.argmax(logits, axis=-1)
df_labelled['pred_sentiment'] = [id2label[p] for p in preds]

print("Classification report (fine-tuned model on labelled data):")
print(classification_report(df_labelled['gold_label'], df_labelled['pred_sentiment'], digits=4))
cm = confusion_matrix(df_labelled['gold_label'], df_labelled['pred_sentiment'], labels=labels)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels)
plt.title("Confusion matrix - fine-tuned model")
plt.show()

In [ ]:
# -------------------------
# 6) Save predictions to CSV for later ABS & SNA
# -------------------------
pred_out = "predictions_finetuned.csv"
df_labelled[['created_at','full_text','clean_text','gold_label','pred_sentiment']].to_csv(pred_out, index=False)
print("Predictions saved to:", pred_out)

In [ ]:
# =========================================================
# ASPECT-BASED SENTIMENT (ABS)
# Two approaches:
# A) Keyword/aspect list (rule-based)
# B) Topic-modeling (LDA) to detect topics (aspects) and assign sentiment per aspect
# =========================================================

# A) Keyword-based approach (recommended quick start)
# - Prepare an 'aspect dictionary': mapping aspect_name -> list of keywords (domain-specific)
# Example: jika dataset terkait produk: price, service, quality, delivery, fitur
ASPECT_DICT = {
    "harga": ["harga","murah","mahal","diskon","ongkir"],
    "layanan": ["pelayanan","layanan","respon","customer","cs","support"],
    "kualitas": ["kualitas","bagus","jelek","rusak","awet","mudah"],
    "fitur": ["fitur","fiturnya","fitur-fiturnya","fiturnya"]
}

# function to extract aspects per tweet and score sentiment for that aspect
# We'll use the fine-tuned model to infer sentiment on sentence/tweet level; for aspect-specific, we can:
# - if aspect keyword present in tweet -> attribute tweet sentiment to that aspect
# - (improvement) split tweet into sentences and score each sentence, attribute to aspects whose keywords appear
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def extract_aspects_keywords(text, aspect_dict=ASPECT_DICT):
    found = []
    for a, kws in aspect_dict.items():
        for kw in kws:
            if re.search(r'\b'+re.escape(kw)+r'\b', text, flags=re.IGNORECASE):
                found.append(a)
                break
    return list(set(found))

# predict sentiment using model wrapper
def predict_text_sentiment(text_list, batch_size=64):
    results = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        enc = tokenizer(batch, truncation=True, padding=True, max_length=128, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model(enc['input_ids'], attention_mask=enc['attention_mask'])
            logits = out.logits.cpu().numpy()
        preds = np.argmax(logits, axis=-1)
        results += [id2label[p] for p in preds]
    return results

# apply aspect extraction and attribute sentiment
df_all = df.copy()   # use original df (unlabeled rows included)
df_all['aspects'] = df_all['clean_text'].apply(lambda t: extract_aspects_keywords(t, ASPECT_DICT))
# filter tweets that have aspects
df_aspect = df_all[df_all['aspects'].map(len)>0].copy()
print("Jumlah tweet yang mengandung aspect keywords:", len(df_aspect))
# predict sentiment for those tweets using fine-tuned model
df_aspect['pred_sentiment'] = predict_text_sentiment(df_aspect['clean_text'].tolist())
# explode aspects and compute counts
df_expl = df_aspect.explode('aspects').reset_index(drop=True)
aspect_sent_counts = df_expl.groupby(['aspects','pred_sentiment']).size().unstack(fill_value=0)
print("Aspect x Sentiment counts (keyword method):\n", aspect_sent_counts)

# visualize aspect sentiment distribution
for aspect in aspect_sent_counts.index:
    vals = aspect_sent_counts.loc[aspect]
    vals = vals.reindex(['positive','neutral','negative']).fillna(0)
    plt.figure(figsize=(5,3))
    sns.barplot(x=vals.index, y=vals.values, palette=['green','gray','red'])
    plt.title(f"Aspect '{aspect}' - sentiment distribution")
    plt.show()

In [ ]:
# B) Topic-modeling (LDA) approach to discover aspects
# - build document-term matrix (use CountVectorizer on clean_text)
V = 8   # number of topics/aspects to try (tune this)
vectorizer = CountVectorizer(max_df=0.9, min_df=5, stop_words=list(stopwords.words('indonesian')))
X = vectorizer.fit_transform(df['clean_text'].astype(str).tolist())
lda = LatentDirichletAllocation(n_components=V, random_state=42, learning_method='batch', max_iter=20)
lda.fit(X)
# show top words per topic
def display_topics(model, feature_names, no_top_words):
    topics = {}
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[:-no_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_indices]
        topics[f"topic_{topic_idx}"] = top_words
        print("Topic %d:" % topic_idx, " ".join(top_words))
    return topics

topics = display_topics(lda, vectorizer.get_feature_names_out(), 12)

# Assign dominant topic/aspect to each document and compute sentiment per topic
doc_topics = lda.transform(X)  # shape (n_docs, V)
dominant_topic = np.argmax(doc_topics, axis=1)
df['dominant_topic'] = dominant_topic
df['topic_sentiment'] = predict_text_sentiment(df['clean_text'].astype(str).tolist())
topic_sent_counts = df.groupby(['dominant_topic','topic_sentiment']).size().unstack(fill_value=0)
print("Topic x Sentiment counts (LDA method):\n", topic_sent_counts)

# visualize topic sentiment
for t in sorted(df['dominant_topic'].unique()):
    if t in topic_sent_counts.index:
        vals = topic_sent_counts.loc[t]
        vals = vals.reindex(['positive','neutral','negative']).fillna(0)
        plt.figure(figsize=(5,3))
        sns.barplot(x=vals.index, y=vals.values, palette=['green','gray','red'])
        plt.title(f"Topic {t} - sentiment distribution")
        plt.show()

# Save aspect outputs
df_expl.to_csv("aspect_keyword_sentiments.csv", index=False)
df[['created_at','full_text','clean_text','dominant_topic','topic_sentiment']].to_csv("aspect_topic_sentiments.csv", index=False)
print("Saved aspect outputs.")

In [ ]:
# =========================================================
#  SNA: build mention/retweet network + analyze centrality & communities per sentiment
# =========================================================
# We'll attempt multiple fallbacks:
# - Preferred: columns 'user_screen_name' / 'user' / 'username' exist => use as source node and parse mentions in text for target nodes
# - If no user column, we'll create co-mention network among mentions inside a tweet (undirected)
# Also attempt to detect retweet edges if retweet info exists (columns like 'retweeted_status_user','retweet_count','is_retweet')

# Detect username/source column
possible_user_cols = [c for c in df.columns if any(k in c.lower() for k in ['user','screen','username','screen_name','from'])]
user_col = possible_user_cols[0] if possible_user_cols else None
print("Detected user column:", user_col)

# parse mentions from text
def parse_mentions(text):
    return re.findall(r'@([A-Za-z0-9_]+)', str(text))

# Build graph
G = nx.DiGraph()  # directed: source -> mention
if user_col:
    df['source_user'] = df[user_col].astype(str)
    for _, row in df.iterrows():
        src = row['source_user']
        mentions = parse_mentions(row.get('full_text',''))  # use raw text so mentions not removed
        for m in mentions:
            G.add_edge(src.lower(), m.lower(), weight=G[src.lower()].get(m.lower(),{}).get('weight',0)+1)
else:
    # fallback: build co-mention undirected graph
    G = nx.Graph()
    for _, row in df.iterrows():
        mentions = parse_mentions(row.get('full_text',''))
        mentions = [m.lower() for m in mentions]
        # build cliques among mentions in same tweet
        for i in range(len(mentions)):
            for j in range(i+1,len(mentions)):
                a,b = mentions[i], mentions[j]
                if G.has_edge(a,b):
                    G[a][b]['weight'] += 1
                else:
                    G.add_edge(a,b,weight=1)

print("Graph nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())

# add node attributes: avg sentiment of tweets posted by that user (if source present)
if user_col:
    # gather tweets per user and compute mean sentiment (map sentiment to numeric)
    sent_map = {'positive':1, 'neutral':0, 'negative':-1}
    user_sents = defaultdict(list)
    for _, r in df.iterrows():
        src = r.get('source_user', '').lower()
        sent = r.get('topic_sentiment') or r.get('pred_sentiment') or r.get('pred_sentiment')  # prefer topic_sentiment then pred_sentiment
        if sent in sent_map:
            user_sents[src].append(sent_map[sent])
    for node in G.nodes():
        vals = user_sents.get(node, [])
        avg = np.mean(vals) if len(vals)>0 else 0.0
        nx.set_node_attributes(G, {node: {'avg_sentiment': avg, 'tweet_count': len(vals)}})
else:
    # for mention-only graph, we cannot compute source sentiment, but can compute mention-level sentiment by scanning tweets where mention appears
    mention_sents = defaultdict(list)
    for _, r in df.iterrows():
        mentions = parse_mentions(r.get('full_text',''))
        sent = r.get('topic_sentiment') or r.get('pred_sentiment')
        for m in mentions:
            mention_sents[m.lower()].append(sent_map.get(sent,0))
    for node in G.nodes():
        vals = mention_sents.get(node, [])
        avg = np.mean(vals) if len(vals)>0 else 0.0
        nx.set_node_attributes(G, {node: {'avg_sentiment': avg, 'mention_count': len(vals)}})

# centrality measures
deg = dict(nx.degree(G))
in_deg = dict(G.in_degree()) if G.is_directed() else deg
out_deg = dict(G.out_degree()) if G.is_directed() else {}
betw = nx.betweenness_centrality(G, k=min(200, G.number_of_nodes())) if G.number_of_nodes()>0 else {}
eigen = nx.eigenvector_centrality_numpy(G) if G.number_of_nodes()>0 and nx.is_connected(G.to_undirected()) else {}
# attach centrality attrs
for n in G.nodes():
    nx.set_node_attributes(G, {n:{
        'degree': deg.get(n,0),
        'in_degree': in_deg.get(n,0),
        'out_degree': out_deg.get(n,0),
        'betweenness': betw.get(n,0),
        'eigenvector': eigen.get(n,0) if n in eigen else 0.0
    }})

# community detection (Louvain)
if G.number_of_nodes()>0:
    # Louvain works on undirected graphs
    partition = community_louvain.best_partition(G.to_undirected())
    # attach partition labels
    for n, p in partition.items():
        nx.set_node_attributes(G, {n: {'community': p}})
    # compute community-level sentiment summary
    comm_sent = defaultdict(list)
    for n in G.nodes():
        attr = G.nodes[n]
        avg_sent = attr.get('avg_sentiment', 0.0)
        comm = attr.get('community', -1)
        comm_sent[comm].append(avg_sent)
    for comm, svals in comm_sent.items():
        print(f"Community {comm}: nodes={len([n for n in G.nodes() if G.nodes[n].get('community')==comm])}, avg_sent={np.mean(svals):.3f}")

In [ ]:
# visualize network: color nodes by community and size by degree, label top nodes
plt.figure(figsize=(12,9))
pos = nx.spring_layout(G, k=0.3, seed=42)
communities = [G.nodes[n].get('community',0) for n in G.nodes()]
deg_vals = np.array([G.nodes[n].get('degree',1) for n in G.nodes()])
# normalize sizes
sizes = 200*(deg_vals / (deg_vals.max() if deg_vals.max()>0 else 1) + 0.1)
unique_comms = list(set(communities))
cmap = plt.get_cmap('tab20')
node_colors = [cmap(unique_comms.index(c) % 20) for c in communities]
nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=node_colors, alpha=0.9)
nx.draw_networkx_edges(G, pos, alpha=0.2, arrows=G.is_directed())
# label top degree nodes
top_nodes = sorted(G.nodes(), key=lambda n: G.nodes[n].get('degree',0), reverse=True)[:20]
nx.draw_networkx_labels(G, pos, labels={n:n for n in top_nodes}, font_size=8)
plt.title("Network graph (nodes colored by Louvain community; size ~ degree)")
plt.axis('off')
plt.show()

# Save graph
nx.write_gexf(G, "mention_retweet_graph.gexf")
print("Graph saved to mention_retweet_graph.gexf (bisa dibuka di Gephi)")

# Save node attributes to csv
nodes_df = pd.DataFrame([{**{'node': n}, **G.nodes[n]} for n in G.nodes()])
nodes_df.to_csv("network_nodes_attributes.csv", index=False)
print("Saved node attributes to network_nodes_attributes.csv")

In [ ]:
# -------------
# End of pipeline
# -------------
print("Pipeline selesai:")
print("- Fine-tuned model ada di:", output_dir)
print("- Prediksi disimpan:", pred_out)
print("- Aspect outputs:", "aspect_keyword_sentiments.csv & aspect_topic_sentiments.csv")
print("- Graph & node attributes:", "mention_retweet_graph.gexf & network_nodes_attributes.csv")